# Task 2: Quantitative Analysis and Technical Indicators

This notebook loads daily OHLCV stock data, validates price fields, computes technical indicators, adds risk/return metrics, and visualizes how indicators relate to price action.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import yfinance as yf

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data_loading import load_price_data, normalize_price_data
from src.indicators import DEFAULT_INDICATOR_PARAMS, add_daily_returns, add_technical_indicators
from src.quant_metrics import add_financial_metrics, add_pynance_metrics
from src.visualization import plot_price_indicators, save_current_figure

sns.set_theme(style='whitegrid')
RAW = PROJECT_ROOT / 'data' / 'raw'
FIGURES = PROJECT_ROOT / 'reports' / 'figures'
FIGURES.mkdir(parents=True, exist_ok=True)

## Load Price Data

Use a local CSV from `data/raw/prices/` when available. If no local file exists, download a reproducible sample with `yfinance`. `auto_adjust=False` keeps the `Adj Close` column required for return calculations.

In [ ]:
TICKER = 'AAPL'
PRICE_FILE = RAW / 'prices' / f'{TICKER}.csv'

if PRICE_FILE.exists():
    prices = load_price_data(PRICE_FILE)
    data_source = f'local CSV: {PRICE_FILE}'
else:
    downloaded = yf.download(
        TICKER,
        start='2020-01-01',
        end='2026-01-01',
        auto_adjust=False,
        progress=False,
    ).reset_index()
    prices = normalize_price_data(downloaded)
    data_source = 'yfinance download: 2020-01-01 to 2026-01-01'

prices['stock'] = TICKER
print(data_source)
prices.head()

## Data Quality Checks

The challenge requires correctly typed OHLCV columns and missing-value handling before indicators are computed.

In [ ]:
numeric_cols = ['Open', 'High', 'Low', 'Close', 'Adj Close', 'Volume']
quality_summary = pd.DataFrame({
    'dtype': prices[['Date'] + numeric_cols].dtypes.astype(str),
    'missing_values': prices[['Date'] + numeric_cols].isna().sum(),
})

print(f'Rows: {len(prices):,}')
print(f'Date range: {prices["Date"].min().date()} to {prices["Date"].max().date()}')
quality_summary

In [ ]:
prices = prices.dropna(subset=['Date', 'Open', 'High', 'Low', 'Close', 'Adj Close', 'Volume']).copy()
prices = prices.sort_values('Date').reset_index(drop=True)
prices[numeric_cols] = prices[numeric_cols].apply(pd.to_numeric, errors='coerce')

assert prices[numeric_cols].notna().all().all()
prices.tail()

## Technical Indicators

Compute the Task 2 indicators with TA-Lib as the preferred engine: SMA, EMA, RSI, and MACD. The notebook records the engine used so the analysis is reproducible even if a local machine falls back to pandas.

Default parameter choices:

- SMA windows: 20 and 50 trading sessions for short- and medium-term trend context
- EMA span: 20 trading sessions for a more responsive trend line
- RSI window: 14 trading sessions with 30/70 oversold/overbought interpretation bands
- MACD: 12-session fast EMA, 26-session slow EMA, and 9-session signal line


In [ ]:
indicators = add_daily_returns(add_technical_indicators(prices, use_talib=True))
indicator_cols = ['sma_20', 'sma_50', 'ema_20', 'rsi_14', 'macd', 'macd_signal', 'macd_hist', 'daily_return_pct']

print(f"Indicator engine: {indicators.attrs.get('indicator_engine')}")
print(f"Indicator parameters: {indicators.attrs.get('indicator_params', DEFAULT_INDICATOR_PARAMS)}")
indicators[['Date', 'Close', 'Adj Close'] + indicator_cols].tail()


## PyNance and Risk/Return Metrics

PyNance is attempted for return/risk metrics. If the local environment has a dependency conflict, equivalent pandas metrics are computed so the analysis remains reproducible.

In [ ]:
pynance_metrics = add_pynance_metrics(prices)
print(pynance_metrics.attrs.get('pynance_status'))

risk_metrics = add_financial_metrics(prices)
risk_metrics[['Date', 'Adj Close', 'cumulative_return', 'rolling_volatility_20', 'rolling_sharpe_20', 'drawdown']].tail()

## Visualize Price, Moving Averages, RSI, and MACD

In [ ]:
fig, ax = plot_price_indicators(indicators, TICKER)
save_current_figure(FIGURES / f'{TICKER}_moving_averages.png')
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(12, 3))
ax.plot(indicators['Date'], indicators['rsi_14'], label='RSI 14', color='tab:purple')
ax.axhline(70, color='tab:red', linestyle='--', linewidth=1, label='Overbought')
ax.axhline(30, color='tab:green', linestyle='--', linewidth=1, label='Oversold')
ax.set_title(f'{TICKER} RSI')
ax.set_ylabel('RSI')
ax.legend()
save_current_figure(FIGURES / f'{TICKER}_rsi.png')
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(indicators['Date'], indicators['macd'], label='MACD', color='tab:blue')
ax.plot(indicators['Date'], indicators['macd_signal'], label='Signal', color='tab:orange')
ax.bar(indicators['Date'], indicators['macd_hist'], label='Histogram', alpha=0.35, color='tab:gray')
ax.axhline(0, color='black', linewidth=0.8)
ax.set_title(f'{TICKER} MACD')
ax.legend()
save_current_figure(FIGURES / f'{TICKER}_macd.png')
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(risk_metrics['Date'], risk_metrics['cumulative_return'] * 100, label='Cumulative return (%)')
ax.plot(risk_metrics['Date'], risk_metrics['drawdown'] * 100, label='Drawdown (%)')
ax.axhline(0, color='black', linewidth=0.8)
ax.set_title(f'{TICKER} Cumulative Return and Drawdown')
ax.set_ylabel('Percent')
ax.legend()
save_current_figure(FIGURES / f'{TICKER}_risk_return.png')
plt.show()

## Interpretation Summary

- TA-Lib is the preferred indicator engine for the final Task 2 workflow; if TA-Lib is unavailable, the code falls back to equivalent pandas calculations and prints the engine used.
- The 20/50-session moving averages show short- versus medium-term trend context and help identify whether the close is trading above or below recent price history.
- The 20-session EMA reacts faster than the SMA and can be used to detect earlier trend changes.
- RSI uses a 14-session window; values above 70 suggest overbought conditions where risk may be reduced, while values below 30 suggest oversold conditions where a rebound watchlist may be reasonable.
- MACD uses 12/26/9 settings; crossovers and histogram sign changes highlight momentum shifts that can be compared later with news sentiment.
- Cumulative return, drawdown, rolling volatility, and Sharpe-style metrics connect the indicators to practical risk management: avoid oversized exposure during high drawdown or volatility periods, and treat sentiment signals as stronger only when technical momentum confirms them.

Data quality note: local CSVs should be preferred for final reproducibility. When `yfinance` is used, record the download date and date range in the final report.
